<a href="https://colab.research.google.com/github/mchlkan/EoDA_Group_Project/blob/main/Projects/CPU_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CPU Baseline — CFPB Consumer Complaints (Machine Learning Assignment)

**Course:** Engineering of Data Analysis
**Runtime:** Colab CPU runtime (no GPU required)

**Purpose**

This notebook establishes the **CPU reference timings** against which `GPU_Solution.ipynb`
is compared. It is one half of a two-notebook benchmark; both notebooks execute the same
pipeline (feature engineering → Logistic Regression → XGBoost) on identical data splits
and with the same hyperparameters — the only difference is the execution engine.

**Pipeline**

The underlying task is binary classification of CFPB consumer complaints (`Consumer disputed?`,
Yes/No) using the same feature engineering and models as the original ML assignment
(`72782_Complaints_Notebook.ipynb`): target-encoded categoricals, narrative-derived text features,
temporal and response-quality flags, fed into
- (1) an L2-regularised Logistic Regression and
- (2) an XGBoost classifier.

**Benchmark design**

1. **Hyperparameter search (context timing).** One full cross-validated search per model on
   the complete training set. Best parameters are persisted to `results/best_params.json`
   and reused by the GPU notebook — this guarantees that the CPU and GPU timings are measuring
   the *same* fitted model, not two different models that each found their own optimum.

2. **Dataset-size sweep.** 3 runs × 5 fractions (10%, 25%, 50%, 75%, 100%) × 6 phases
   (`fe_train`, `fe_test`, `lr_fit`, `lr_predict`, `xgb_fit`, `xgb_predict`) = 90 timing rows,
   written to `results/timings_cpu.csv`. Phase-level granularity lets the downstream analysis
   report training vs. inference speedups separately, as required by the assignment brief.

**Outputs**

- `results/best_params.json` — best hyperparameters + CV context timings (consumed by GPU notebook)
- `results/timings_cpu.csv` — 90-row long-format timing log (consumed by analysis notebook)

## Table of Contents

1. [Setup & Imports](#1-setup--imports)
2. [Data Loading & Train/Test Split](#2-data-loading--traintest-split)
3. [Feature Engineering](#3-feature-engineering)
4. [Hyperparameter Search (Context Timing)](#4-hyperparameter-search-context-timing)
    - 4.1 [Logistic Regression — GridSearchCV](#41-logistic-regression--gridsearchcv)
    - 4.2 [XGBoost — RandomizedSearchCV](#42-xgboost--randomizedsearchcv)
    - 4.3 [Persist Best Parameters](#43-persist-best-parameters)
5. [Dataset-Size Sweep](#5-dataset-size-sweep)
    - 5.1 [Timing Utility](#51-timing-utility)
    - 5.2 [Fit/Predict Sweep](#52-fitpredict-sweep)
    - 5.3 [HPO Sweep](#53-hpo-sweep)
    - 5.4 [Sanity Check](#54-sanity-check)

## 1. Setup & Imports

In [2]:
# Imports
import numpy as np
import pandas as pd
import time
import json
import os

from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# Fix random seed for reproducibility
np.random.seed(42)

## 2. Data Loading & Train/Test Split

We use the same CFPB dataset and the same 80/20 stratified split
(`random_state=42`) as the original ML notebook. Both CPU and GPU
notebooks must produce identical splits — this is what makes the
timing comparison apples-to-apples. The target (`Consumer disputed?`)
is heavily imbalanced (~21% positive), so the split is stratified
on `y` to preserve the class ratio in both sets.

In [3]:
# Load the training CSV. Adjust DATA_PATH to wherever your copy lives.
# Default: Google Drive (Colab). Falls back to local path if Drive isn't available.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/complaints_training.csv'
except ImportError:
    DATA_PATH = 'complaints_training.csv'

df_raw = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df_raw):,} rows × {df_raw.shape[1]} columns from {DATA_PATH}')

Mounted at /content/drive
Loaded 321,430 rows × 18 columns from /content/drive/MyDrive/complaints_training.csv


In [4]:
# Features: everything except the ID column and the target itself.
# Target: binary-encode 'Consumer disputed?' to {0, 1}.
X = df_raw.drop(columns=['Complaint ID', 'Consumer disputed?'])
y = df_raw['Consumer disputed?'].map({'Yes': 1, 'No': 0})

# 80/20 stratified split
# Seed is the same as in the original ML notebook resulting in same splits
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f'Train: {len(X_train_raw):,} rows  |  positive rate: {y_train.mean():.3f}')
print(f'Test:  {len(X_test_raw):,} rows  |  positive rate: {y_test.mean():.3f}')

Train: 257,144 rows  |  positive rate: 0.199
Test:  64,286 rows  |  positive rate: 0.199


## 3. Feature Engineering

All feature construction is consolidated into a single `feature_engineering()`
function, identical to the one in the original ML notebook. The GPU notebook
reimplements this function as `feature_engineering_gpu()` using cuDF primitives;
a correctness check in the GPU notebook verifies that both produce numerically
equivalent output column-by-column.

**Output features (~38 columns) fall into five groups:**

- **Temporal** — response lag (days between receipt and forwarding), received
  year, day of week.
- **Response-quality flags** — regex-derived binaries on the company response
  text (monetary relief, explanation only, etc.), timely-response flag,
  interactions.
- **Narrative-derived** — word count, char count, average word length, presence
  of legal/escalation terms (`attorney`, `lawsuit`, `fraud`, …).
- **Metadata flags** — consent provided/withdrawn, sub-product/sub-issue
  present, vulnerability tags (older Americans, servicemembers).
- **Encoded categoricals** — smoothed target encoding (with additive smoothing,
  α=20) for high-cardinality columns (`Product`, `Company`, `State`, `ZIP3`,
  interactions), plus frequency encoding for `Company`.

**Train vs. inference mode.** The function has two modes controlled by
`fit_encoders`:
- `fit_encoders=None` — **training mode**: encoders are fit on the incoming
  data (computes per-category target means etc.) and the transformed frame is
  returned alongside the fitted encoders.
- `fit_encoders=<dict>` — **inference mode**: pre-fitted encoders are applied
  as-is, with no re-fitting. This prevents test-set leakage.

In the sweep below, `fe_train` calls the function in training mode on the
fraction-sized training subset; `fe_test` applies the resulting encoders to
the full (fraction-independent) test set.

In [5]:
def feature_engineering(X, y=None, fit_encoders=None):
    encoders = {} if fit_encoders is None else fit_encoders
    training = fit_encoders is None
    raw = X.copy()
    for col in ['Complaint ID', 'Consumer disputed?']:
        if col in raw.columns:
            raw = raw.drop(columns=[col])
    out = pd.DataFrame(index=raw.index)

    date_rec  = pd.to_datetime(raw['Date received'],        errors='coerce')
    date_sent = pd.to_datetime(raw['Date sent to company'], errors='coerce')
    out['response_lag_days'] = (date_sent - date_rec).dt.days.clip(0, 120).fillna(0).astype(int)
    out['received_year']     = date_rec.dt.year.fillna(2015).astype(int)
    out['day_of_week']       = date_rec.dt.dayofweek.fillna(0).astype(int)

    resp = raw['Company response to consumer'].fillna('Missing')
    out['company_response_monetary']       = resp.str.contains('monetary',       case=False).astype(int)
    out['company_response_non_monetary']   = resp.str.contains('non-monetary',   case=False).astype(int)
    out['company_response_explanation']    = resp.str.contains('explanation',    case=False).astype(int)
    out['company_response_without_relief'] = resp.str.contains('without relief', case=False).astype(int)
    out['company_response_in_progress']    = resp.str.contains('in progress',    case=False).astype(int)
    out['company_response_is_relief']      = (
        (out['company_response_monetary'] == 1) |
        (out['company_response_non_monetary'] == 1)
    ).astype(int)
    out['timely_response']     = raw['Timely response?'].fillna('No').eq('Yes').astype(int)
    out['closed_x_timely']     = out['company_response_is_relief'] * out['timely_response']
    out['has_public_response'] = raw['Company public response'].notna().astype(int)

    narr = raw['Consumer complaint narrative'].fillna('')
    out['has_narrative']       = (narr != '').astype(int)
    out['word_count']          = narr.apply(lambda x: len(x.split()) if x else 0)
    out['char_count']          = narr.str.len().fillna(0).astype(int)
    out['avg_word_length']     = narr.apply(
        lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0
    )
    out['narrative_short']     = (out['word_count'].between(1, 19)).astype(int)
    out['narrative_long']      = (out['word_count'] > 200).astype(int)
    esc = 'attorney|lawyer|lawsuit|legal action|sue|court|cfpb|regulation|violation|fraud|illegal|discriminat'
    out['escalation_term_count'] = narr.str.lower().str.count(esc).fillna(0).astype(int)
    out['has_escalation_terms']  = (out['escalation_term_count'] > 0).astype(int)
    out['narrative_x_no_relief'] = out['has_narrative'] * (1 - out['company_response_is_relief'])

    consent = raw['Consumer consent provided?'].fillna('Missing')
    out['consent_provided']  = consent.eq('Consent provided').astype(int)
    out['consent_withdrawn'] = consent.eq('Consent withdrawn').astype(int)
    out['has_subproduct']    = raw['Sub-product'].notna().astype(int)
    out['has_subissue']      = raw['Sub-issue'].notna().astype(int)
    tags = raw['Tags'].fillna('')
    out['tags_any']          = (tags != '').astype(int)
    out['is_older_american'] = tags.str.contains('Older American', case=False).astype(int)
    out['is_servicemember']  = tags.str.contains('Servicemember',  case=False).astype(int)

    def target_encode(series, name, smoothing=20):
        if training:
            tmp = pd.DataFrame({'val': series.values, 'target': y.values}, index=series.index)
            agg = tmp.groupby('val')['target'].agg(['mean', 'count'])
            global_mean = y.mean()
            smooth = (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)
            encoders[name] = {'map': smooth.to_dict(), 'global_mean': global_mean}
        return series.map(encoders[name]['map']).fillna(encoders[name]['global_mean'])

    def freq_encode(series, name):
        if training:
            encoders[name] = series.value_counts().to_dict()
        return series.map(encoders[name]).fillna(1)

    out['Product_te']           = target_encode(raw['Product'].fillna('Missing'),                      'Product_te')
    out['Issue_te']             = target_encode(raw['Issue'].fillna('Missing'),                        'Issue_te')
    out['State_te']             = target_encode(raw['State'].fillna('Missing'),                        'State_te')
    out['Company_te']           = target_encode(raw['Company'].fillna('Missing'),                      'Company_te')
    out['Sub_product_te']       = target_encode(raw['Sub-product'].fillna('Missing'),                  'Sub_product_te')
    out['Sub_issue_te']         = target_encode(raw['Sub-issue'].fillna('Missing'),                    'Sub_issue_te')
    out['Submitted_via_te']     = target_encode(raw['Submitted via'].fillna('Missing'),                'Submitted_via_te')
    out['company_response_te']  = target_encode(raw['Company response to consumer'].fillna('Missing'), 'company_response_te')
    out['zip3_region_te']       = target_encode(
        raw['ZIP code'].fillna('000').astype(str).str[:3],                                             'zip3_region_te')
    prod_x_channel = raw['Product'].fillna('Missing') + '_' + raw['Submitted via'].fillna('Missing')
    out['product_x_channel_te'] = target_encode(prod_x_channel, 'product_x_channel_te')
    prod_x_issue = raw['Product'].fillna('Missing') + '_' + raw['Issue'].fillna('Missing')
    out['product_issue_te']     = target_encode(prod_x_issue, 'product_issue_te')
    out['company_volume']       = freq_encode(raw['Company'].fillna('Missing'), 'company_volume')
    out['response_te_x_relief'] = out['company_response_te'] * out['company_response_is_relief']
    return out, encoders


The cell below runs feature engineering **once** on the full training set. Its
output is consumed only by the hyperparameter search in section 4 — so the CV
sees the full 257k-row training set when picking best parameters. The
dataset-size sweep in section 5 re-runs feature engineering from scratch on
each fraction (that's the whole point of timing FE as a function of data size),
so nothing here is reused there.

In [6]:
# Fit encoders on the full training set; apply to the test set with no refitting.
# Used only by the hyperparameter search in section 4.
# The dataset-size sweep in section 5 rebuilds features per fraction and does not reuse these objects.
X_train_enc, encoders = feature_engineering(X_train_raw, y=y_train)
X_test_enc, _         = feature_engineering(X_test_raw, fit_encoders=encoders)

assert X_train_enc.isnull().sum().sum() == 0, 'Nulls in training features — check FE'
assert X_test_enc.isnull().sum().sum()  == 0, 'Nulls in test features — check FE'
print(f'Features: {X_train_enc.shape[1]}  |  train rows: {len(X_train_enc):,}  |  test rows: {len(X_test_enc):,}')

Features: 41  |  train rows: 257,144  |  test rows: 64,286


## 4. Hyperparameter Search (Context Timing)

Both models are tuned with stratified 5-fold cross-validation on the full
training set. The resulting best parameters are saved to
`results/best_params.json` and reused by the GPU notebook. This guarantees
that the CPU and GPU sweeps in section 5 are timing the same model, not
two differently-tuned models.

The CV runs themselves are also timed (as "context" timings — separate
from the main sweep) and later included as a dedicated timing phase in
section 5.3, where they are compared against their GPU counterparts to
produce an HPO speedup figure for the report.


### 4.1 Logistic Regression — GridSearchCV

**Model.** L2-regularised Logistic Regression with class-balanced weights.
Linear classifier that models the log-odds of dispute as a weighted sum of
features; serves as our interpretable baseline against which the non-linear
XGBoost must prove its worth.

**Grid.** Five values of `C` (inverse regularisation strength) spanning four
orders of magnitude. `penalty='l2'` only — L1 was dropped for cross-framework
consistency: cuML's Logistic Regression uses a quasi-Newton (QN) solver that
does not cleanly support L1 regularisation, so restricting the CPU grid to
L2 guarantees that whichever parameters win here will produce a numerically
equivalent model on the GPU.

**Scoring: F1.** The target is imbalanced (~21% positive), so accuracy is
misleading and ROC-AUC, while useful as a summary, doesn't capture the
precision–recall trade-off at any operating threshold. F1 was used in the
original assignment and we preserve that choice here.

**Solver: lbfgs.** Quasi-Newton, sklearn's default for L2 penalties and the
closest CPU analogue to cuML's QN solver.

**`n_jobs=1`** — CV folds run serially rather than in parallel. We default
to serial CV because XGBoost (tuned next) already parallelises internally
via OpenMP, and running parallel CV on top produces noisy timings on the
2-core Colab free tier. The same `n_jobs=1` is used for the XGB search so
the two context timings are directly comparable.

In [7]:
# LR GridSearchCV — one-off, full training set.
# Produces best hyperparameters (saved to JSON) and a context timing
# (later benchmarked vs. GPU in section 5.3 as the HPO speedup).

lr_pipeline = Pipeline([
    ('imputer',    SimpleImputer(strategy='median')),
    ('scaler',     StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# L2-only grid: required for cross-framework equivalence with cuML's QN solver.
param_grid_lr = {
    'classifier__C':            [0.001, 0.01, 0.1, 1, 10],
    'classifier__penalty':      ['l2'],
    'classifier__class_weight': ['balanced'],
    'classifier__solver':       ['lbfgs'],
}

cv_lr = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search_lr = GridSearchCV(
    lr_pipeline, param_grid_lr,
    cv=cv_lr, scoring='f1', n_jobs=1, verbose=1,
)

t0 = time.perf_counter()
search_lr.fit(X_train_enc, y_train)
t_lr_cv = time.perf_counter() - t0

# Score the best model on the held-out test set. This AUC is what the GPU
# notebook's cuML LR will be compared against (correctness assert).
y_prob_lr = search_lr.best_estimator_.predict_proba(X_test_enc)[:, 1]
auc_lr    = roc_auc_score(y_test, y_prob_lr)

n_fits = len(search_lr.cv_results_['params']) * cv_lr.get_n_splits()
print(f'Best LR params: {search_lr.best_params_}')
print(f'CV F1:          {search_lr.best_score_:.4f}')
print(f'Test ROC-AUC:   {auc_lr:.4f}')
print(f'[CONTEXT] LR GridSearchCV ({n_fits} fits): {t_lr_cv:.1f}s  ({t_lr_cv/60:.1f} min)')

Fitting 5 folds for each of 5 candidates, totalling 25 fits


KeyboardInterrupt: 

### 4.2 XGBoost — RandomizedSearchCV

**Model.** Gradient-boosted decision trees (histogram-based splitting).
Non-linear and non-additive — captures interactions between features that
the linear LR cannot. XGBoost is the production workhorse of the original
ML notebook and the strongest candidate for GPU acceleration in this
pipeline: its `device='cuda'` setting is a near drop-in for `device='cpu'`,
so CPU and GPU runs execute the same algorithm on different hardware.

**Grid.** Nine hyperparameters spanning tree complexity (`max_depth`,
`min_child_weight`), regularisation (`reg_alpha`, `reg_lambda`), stochasticity
(`subsample`, `colsample_bytree`), imbalance handling (`scale_pos_weight`),
and the boosting budget (`n_estimators`, `learning_rate`). The full grid has
~46,000 combinations, so an exhaustive `GridSearchCV` is infeasible; we use
`RandomizedSearchCV` with 60 samples — 60 × 5 folds = **300 fits**, enough
to probe the space without dominating the notebook runtime.

**`tree_method='hist'`, `device='cpu'`.** Histogram-based tree building is
the algorithm that has a direct GPU equivalent in XGBoost (`device='cuda'`
swaps the backend with no change to the algorithm). This makes the CPU vs.
GPU comparison a clean apples-to-apples hardware swap.

**Scoring and `n_jobs`.** Identical to the LR search (F1, `n_jobs=1`) for
the same reasons — see 4.1. XGBoost additionally uses OpenMP internally to
parallelise tree construction across CPU cores, so a single `.fit()` already
saturates available cores on Colab; parallelising CV on top would only
introduce scheduler noise.

In [ ]:
# XGBoost RandomizedSearchCV — one-off, full training set.
# Produces best hyperparameters (saved to JSON) and a context timing

xgb_pipeline = Pipeline([
    ('imputer',    SimpleImputer(strategy='median')),
    # tree_method='hist' + device='cpu' is the CPU twin of device='cuda'.
    # Same algorithm, different backend — enables apples-to-apples GPU compare.
    ('classifier', XGBClassifier(
        random_state=42, eval_metric='logloss',
        tree_method='hist', device='cpu',
    ))
])

param_grid_xgb = {
    'classifier__n_estimators':     [400, 500, 600, 800],
    'classifier__max_depth':        [4, 5, 6],
    'classifier__learning_rate':    [0.02, 0.03, 0.05],
    'classifier__scale_pos_weight': [3.5, 4, 4.5],
    'classifier__min_child_weight': [5, 10, 15],
    'classifier__subsample':        [0.55, 0.6, 0.65, 0.7],
    'classifier__colsample_bytree': [0.6, 0.65, 0.7],
    'classifier__reg_alpha':        [0, 0.1, 1],
    'classifier__reg_lambda':       [3, 5, 7],
}

cv_xgb = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search_xgb = RandomizedSearchCV(
    xgb_pipeline, param_grid_xgb,
    n_iter=60, cv=cv_xgb, scoring='f1',
    random_state=42, n_jobs=1, verbose=1,
)

t0 = time.perf_counter()
search_xgb.fit(X_train_enc, y_train)
t_xgb_cv = time.perf_counter() - t0

# Score best model on held-out test set. This AUC is the correctness reference
# for the GPU XGB search (asserted in GPU_Solution.ipynb within 0.005 tolerance).
y_prob_xgb = search_xgb.best_estimator_.predict_proba(X_test_enc)[:, 1]
auc_xgb    = roc_auc_score(y_test, y_prob_xgb)

n_fits = search_xgb.n_iter * cv_xgb.get_n_splits()
print(f'Best XGB params: {search_xgb.best_params_}')
print(f'CV F1:           {search_xgb.best_score_:.4f}')
print(f'Test ROC-AUC:    {auc_xgb:.4f}')
print(f'[CONTEXT] XGB RandomizedCV ({n_fits} fits): {t_xgb_cv:.1f}s  ({t_xgb_cv/60:.1f} min)')

### 4.3 Persist Best Parameters

The best hyperparameters, test-set AUCs, and context timings are written to
`results/best_params.json`. The GPU notebook reads this file in its section 4
and uses:
- the hyperparameters to configure identical models in the GPU sweep,
- the AUCs as correctness references (`abs(auc_gpu - auc_cpu) < tol`),
- the context timings as one data point in the HPO speedup analysis.

The `classifier__` prefix from the sklearn `Pipeline` is stripped so the
dict can be splatted directly into the GPU model constructors.

In [ ]:
# Persist best hyperparameters, AUCs, and context timings for GPU_Solution.ipynb.
os.makedirs('results', exist_ok=True)

# Strip the sklearn Pipeline prefix so the dict can be **unpacked into
# model constructors on the GPU side without renaming.
best_lr_params  = {k.replace('classifier__', ''): v for k, v in search_lr.best_params_.items()}
best_xgb_params = {k.replace('classifier__', ''): v for k, v in search_xgb.best_params_.items()}

best_params = {
    'lr':  best_lr_params,
    'xgb': best_xgb_params,
    'auc': {
        'lr':  float(auc_lr),
        'xgb': float(auc_xgb),
    },
    'cv_context_seconds': {
        'lr_gridsearch':        t_lr_cv,
        'xgb_randomizedsearch': t_xgb_cv,
    },
}

out_path = 'results/best_params.json'
with open(out_path, 'w') as f:
    json.dump(best_params, f, indent=2)

print(f'Saved {out_path}')
print(json.dumps(best_params, indent=2))

## 5. Dataset-Size Sweep

The main benchmark. For each of 5 fractions (10%, 25%, 50%, 75%, 100% of the
training set), we re-run the full pipeline and time six phases: feature
engineering (train and test), LR fit and predict, XGBoost fit and predict.
Three outer runs are collected to estimate timing variance. Total:
**3 × 5 × 6 = 90 timing rows**, written to `results/timings_cpu.csv`.

Section 5.3 adds an additional HPO (hyperparameter-search) sweep at a subset
of fractions — this is the data behind the report's HPO speedup figure.

### 5.1 Timing Utility

All timed calls funnel through `log_row()` — one function, one schema, zero
chance of a column drift that would break the join against `timings_gpu.csv`
in the analysis notebook. The GPU notebook uses the identical helper.

**Schema:** `device`, `fraction`, `n_rows`, `phase`, `run_idx`, `seconds`, `notes`.

In [8]:
# Timing utility. Every timed call in the sweep below goes through log_row()
# to guarantee a consistent schema across CPU and GPU notebooks.

_rows = []

def log_row(device, fraction, n_rows, phase, run_idx, seconds, notes=''):
    """Append one timing record to the in-memory buffer.

    Columns: device, fraction, n_rows, phase, run_idx, seconds, notes.
    Buffer is flushed to CSV after every outer run of the sweep.
    """
    _rows.append({
        'device':   device,
        'fraction': fraction,
        'n_rows':   n_rows,
        'phase':    phase,
        'run_idx':  run_idx,
        'seconds':  seconds,
        'notes':    notes,
    })

### 5.2 Fit/Predict Sweep

Three outer runs × five fractions × six phases. Each iteration builds a fresh
model from the best hyperparameters loaded in section 4, runs feature
engineering on the fraction's training subset, fits and predicts with LR
and XGBoost on that subset, and logs wall-clock time for every phase.

**Design choices to note:**

- **Nested subsets.** All fractions use `random_state=42`, so the 10% subset
  is a subset of the 25%, which is a subset of the 50%, etc. This isolates
  dataset-size effects from sampling effects — we are measuring *"how does
  time scale with n"*, not *"how does time vary across different samples
  of size n"*.
- **Three runs with the same seed.** Runs time the **same subset** three
  times to capture pure timing variance (CPU scheduling, cache state,
  memory pressure). Sampling variance is deliberately excluded — it's not
  what this benchmark is about.
- **Fresh model per iteration.** LR and XGB are reconstructed inside the
  loop rather than fit-then-refit. Prevents any warm-start, JIT, or internal
  state from previous iterations contaminating the timing.
- **Fraction-specific `fe_test` encoders.** The test set is feature-engineered
  using encoders fit on the fraction-sized training subset, not the full
  training set. This reflects real production behaviour (you only have the
  training data you have) and lets `fe_test` capture pure FE throughput as
  encoder-dict size grows with fraction.
- **Best hyperparameters are fixed.** Loaded from `best_params.json`; not
  re-tuned per fraction. Re-tuning would conflate model-performance
  variance with timing variance. Timing the *search itself* as a function
  of fraction is done separately in section 5.3.
- **CSV flushed after each outer run.** Colab sessions can die mid-sweep
  (idle timeout, GPU memory spike, preemption). Writing after each full
  pass means at least one complete run is recoverable.

In [9]:
# Dataset-size sweep (fit/predict phases).
# 3 runs × 5 fractions × 6 phases = 90 rows, written to results/timings_cpu.csv.
# See markdown above for design choices.

FRACS = [0.10, 0.25, 0.50, 0.75, 1.00]
RUNS  = 3

for run_idx in range(RUNS):
    for frac in FRACS:
        # Nested subsets: same seed across fractions means 10% ⊂ 25% ⊂ … ⊂ 100%.
        X_sub = X_train_raw.sample(frac=frac, random_state=42)
        y_sub = y_train.loc[X_sub.index]
        n = len(X_sub)

        # --- Feature engineering (train) -------------------------------
        t0 = time.perf_counter()
        X_sub_enc, enc_sub = feature_engineering(X_sub, y=y_sub)
        log_row('cpu', frac, n, 'fe_train', run_idx, time.perf_counter() - t0)

        # --- Feature engineering (test) --------------------------------
        # Uses fraction-specific encoders — reflects real production usage.
        t0 = time.perf_counter()
        X_test_sub, _ = feature_engineering(X_test_raw, fit_encoders=enc_sub)
        log_row('cpu', frac, n, 'fe_test', run_idx, time.perf_counter() - t0)

        # --- Logistic Regression ---------------------------------------
        # Fresh pipeline per iteration → no state leakage across runs.
        lr = Pipeline([
            ('imputer',    SimpleImputer(strategy='median')),
            ('scaler',     StandardScaler()),
            ('classifier', LogisticRegression(
                **best_lr_params, random_state=42, max_iter=1000
            )),
        ])
        t0 = time.perf_counter()
        lr.fit(X_sub_enc, y_sub)
        log_row('cpu', frac, n, 'lr_fit', run_idx, time.perf_counter() - t0)

        t0 = time.perf_counter()
        lr.predict_proba(X_test_sub)
        log_row('cpu', frac, n, 'lr_predict', run_idx, time.perf_counter() - t0)

        # --- XGBoost ---------------------------------------------------
        # device='cpu' is the default, stated here to mirror device='cuda' in GPU notebook.
        xgb = Pipeline([
            ('imputer',    SimpleImputer(strategy='median')),
            ('classifier', XGBClassifier(
                **best_xgb_params,
                tree_method='hist', device='cpu',
                random_state=42, eval_metric='logloss',
            )),
        ])
        t0 = time.perf_counter()
        xgb.fit(X_sub_enc, y_sub)
        log_row('cpu', frac, n, 'xgb_fit', run_idx, time.perf_counter() - t0)

        t0 = time.perf_counter()
        xgb.predict_proba(X_test_sub)
        log_row('cpu', frac, n, 'xgb_predict', run_idx, time.perf_counter() - t0)

        print(f'run={run_idx} frac={frac:.0%} n={n:,} done')

    # Flush after each outer run — recoverable if Colab dies mid-sweep.
    pd.DataFrame(_rows).to_csv('results/timings_cpu.csv', index=False)
    print(f'[run {run_idx}] Saved results/timings_cpu.csv  ({len(_rows)} rows so far)')

KeyboardInterrupt: 

### 5.3 HPO Sweep

The fit/predict sweep in 5.2 measures what happens **after** hyperparameters
are chosen. But cross-validated hyperparameter search is itself an
embarrassingly parallel workload — `n_iter × k_folds` independent model
fits — and one of the key scaling axes that GPU frameworks accelerate
(Lecture 3, slide 35: *"Multiple independent jobs — hyperparameter sweeps,
cross-validation folds — that could run in parallel"*).

This section re-runs the hyperparameter searches from 4.1 (LR) and 4.2 (XGB)
at multiple dataset fractions, adding two new phases to the timing log:

- `lr_cv` — full L2 LR grid search, 5 folds, 25 fits per fraction.
- `xgb_cv` — randomized search with 60 samples, 5 folds, 300 fits per fraction.

**Fractions measured:**
- LR: 25%, 50%, 100% — 3 points, CV is cheap (~seconds per fit).
- XGB: 50%, 100% — 2 points, CV is the dominant runtime.

**Single run per fraction.** The HPO context timings in 4.1/4.2 already
establish baseline values at 100%; this section's contribution is the
*curve*, not the variance estimate. The GPU notebook mirrors this exactly.

In [10]:
# HPO sweep: time the full hyperparameter search at multiple fractions.
# Each iteration replays the CV from section 4 on a fraction of the training set.

LR_FRACS  = [0.25, 0.50, 1.00]
XGB_FRACS = [0.50, 1.00]

# --- LR HPO sweep ------------------------------------------------------
for frac in LR_FRACS:
    X_sub = X_train_raw.sample(frac=frac, random_state=42)
    y_sub = y_train.loc[X_sub.index]
    n = len(X_sub)

    # Rebuild features on the fraction — matches what CV in 4.1 did.
    X_sub_enc, _ = feature_engineering(X_sub, y=y_sub)

    # Re-build the exact same search object as 4.1 (fresh for isolation).
    pipe = Pipeline([
        ('imputer',    SimpleImputer(strategy='median')),
        ('scaler',     StandardScaler()),
        ('classifier', LogisticRegression(random_state=42, max_iter=1000)),
    ])
    search = GridSearchCV(
        pipe, param_grid_lr,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='f1', n_jobs=1, verbose=0,
    )
    t0 = time.perf_counter()
    search.fit(X_sub_enc, y_sub)
    log_row('cpu', frac, n, 'lr_cv', run_idx=0,
            seconds=time.perf_counter() - t0,
            notes=f'{len(search.cv_results_["params"]) * 5} fits')
    print(f'lr_cv  frac={frac:.0%}  n={n:,}  done')

# --- XGB HPO sweep -----------------------------------------------------
for frac in XGB_FRACS:
    X_sub = X_train_raw.sample(frac=frac, random_state=42)
    y_sub = y_train.loc[X_sub.index]
    n = len(X_sub)

    X_sub_enc, _ = feature_engineering(X_sub, y=y_sub)

    pipe = Pipeline([
        ('imputer',    SimpleImputer(strategy='median')),
        ('classifier', XGBClassifier(
            random_state=42, eval_metric='logloss',
            tree_method='hist', device='cpu',
        )),
    ])
    search = RandomizedSearchCV(
        pipe, param_grid_xgb,
        n_iter=60, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='f1', random_state=42, n_jobs=1, verbose=0,
    )
    t0 = time.perf_counter()
    search.fit(X_sub_enc, y_sub)
    log_row('cpu', frac, n, 'xgb_cv', run_idx=0,
            seconds=time.perf_counter() - t0,
            notes=f'{search.n_iter * 5} fits')
    print(f'xgb_cv frac={frac:.0%}  n={n:,}  done')

# Flush HPO rows to CSV.
pd.DataFrame(_rows).to_csv('results/timings_cpu.csv', index=False)
print(f'Saved results/timings_cpu.csv  ({len(_rows)} rows total)')

SyntaxError: incomplete input (142994081.py, line 56)

### 5.4 Sanity Check

Reload the CSV and verify row count. Expected:
- 90 rows from the fit/predict sweep (3 runs × 5 fractions × 6 phases)
- 3 rows from the LR HPO sweep
- 2 rows from the XGB HPO sweep
- **Total: 95 rows**

A median-time table per phase × fraction is printed as a quick eyeball of
the results before handing off to the analysis notebook.

In [ ]:
# Sanity check: reload CSV, verify row count, preview median timings.
df_cpu = pd.read_csv('results/timings_cpu.csv')

expected_rows = (RUNS * len(FRACS) * 6) + len(LR_FRACS) + len(XGB_FRACS)  # 90 + 3 + 2 = 95
assert len(df_cpu) == expected_rows, (
    f'Row count mismatch: got {len(df_cpu)}, expected {expected_rows}'
)
print(f'Row count OK: {len(df_cpu)} rows')

print('\nPhase coverage:')
print(df_cpu['phase'].value_counts().sort_index().to_string())

print('\nMedian seconds per phase × fraction:')
print(
    df_cpu.groupby(['phase', 'fraction'])['seconds']
          .median()
          .unstack()
          .round(3)
          .to_string()
)

In [ ]:
# printing the total sweep time
import datetime
total_sweep_time = df_cpu['seconds'].sum()
print(f'Total measured time (all phases, all runs): {datetime.timedelta(seconds=int(total_sweep_time))}')